# 第 30 课：生产部署——容器、并发、监控、灰度与验收

模型能够运行只是部署的起点。本课建立上线前的完整检查清单。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 量化与部署 |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 29 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | 容器复现、并发线程、监控灰度回滚 |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：容器复现、并发线程、监控灰度回滚。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


<!-- course-bridge-v3 -->
## 知识接力：先取回旧知识，再进入本课

### 3 分钟闭卷回忆

在新 Markdown cell 中回答，**不要先翻前文**：PyTorch 基线输出；shape/dtype/dynamic axes；流式 cache；P50/P99 与 RTF。

- 三项都能用“含义 + 单位/shape + 一个数字例子”回答：进入本课。
- 能回答两项：学习本课，但把缺口记入 `LEARNING_LOG.md`。
- 只能回答零到一项：先回到 [上一课](29_WebSocket流式服务_会话状态与PGS.ipynb)与[唯一学习路径](../LEARNING_PATH.md)，做一次最小实验；不要靠继续看新术语掩盖断点。

### 本课接口契约

```text
输入：冻结模型与数值基线、目标硬件、输入输出/状态协议
  ↓ 本课要学会的变换、状态或判断
输出：有一致性、性能、并发隔离和回滚证据的部署产物
```

学完后必须能解释：输入的哪个单位/shape/状态若丢失，会让输出“仍能运行却语义错误”。


In [ ]:
from pathlib import Path
import time
import sys
import numpy as np
import matplotlib.pyplot as plt

def find_root():
    here=Path.cwd().resolve()
    for p in [here,*here.parents]:
        if (p/"pyproject.toml").exists(): return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT=find_root(); ARTIFACTS=ROOT/"artifacts";ARTIFACTS.mkdir(exist_ok=True)
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
plt.rcParams["figure.figsize"]=(11,4)
print("项目根目录:",ROOT)

from concurrent.futures import ThreadPoolExecutor
import onnxruntime as ort
from deployment.model import FEATURE_DIM,CHUNK_FRAMES,CACHE_FRAMES
model_path=ARTIFACTS/"streaming_ctc_demo.int8.onnx"
session=ort.InferenceSession(str(model_path),providers=["CPUExecutionProvider"])

## 1. 并发不是简单增加线程

ORT 自身有 intra-op/inter-op 线程池，服务框架也有 worker/线程。层层都开满会发生 oversubscription，造成 P99 抖动。应在固定 CPU 配额下联合调参。

In [ ]:
rng=np.random.default_rng(4);x=rng.normal(size=(1,CHUNK_FRAMES,FEATURE_DIM)).astype(np.float32);c=np.zeros((1,CACHE_FRAMES,FEATURE_DIM),np.float32)
def one(_):
    t=time.perf_counter();session.run(None,{"frames":x,"cache":c});return (time.perf_counter()-t)*1000
for workers in [1,2,4,8]:
    with ThreadPoolExecutor(max_workers=workers) as pool: times=np.array(list(pool.map(one,range(300))))
    print("workers",workers,"throughput req/s",len(times)/(times.sum()/1000),"P50/P99 ms",np.percentile(times,[50,99]))

上面的 throughput 算法是教学近似：并发请求有重叠，严谨吞吐应使用整批 wall-clock 时间；延迟仍按每请求统计。请在练习中修正它。

## 2. Docker 构建

项目已经准备 `deployment/Dockerfile`：

```powershell
docker build -f deployment/Dockerfile -t learn-asr:0.1 .
docker run --rm -p 8000:8000 learn-asr:0.1
```

镜像应固定 lockfile、模型 checksum、运行时版本和非 root 用户。教学 Dockerfile 保持简洁，生产版还需要最小权限和镜像扫描。

## 3. 监控四类信号

- Traffic：连接数、音频秒数、请求率、并发；
- Errors：协议错误、模型异常、断线、OOM；
- Latency：first partial/stable/final、P50/P90/P99、队列等待；
- Saturation：CPU/GPU、内存、线程池、队列、网络。

ASR 还必须监控 RTF、空结果率、平均输出长度、CER/WER 抽样和热词误触发。

## 4. 灰度与回滚

新模型不能只比较总体 WER：需要固定回归集、领域分桶、量化版对照、shadow traffic、少量灰度、版本化指标和一键回滚。服务事件应携带 model/config/LM/graph 版本，才能追查问题。

## 5. 安全与隐私

语音可能含敏感信息。必须考虑 TLS、认证授权、日志脱敏、音频保留策略、地域与合规、依赖供应链，以及对超长输入、连接洪泛和恶意 ONNX 模型的限制。

## 最终部署验收题

1. 量化部署必须比较哪三类结果？
2. RTF、首字延迟和 P99 为什么必须同时报告？
3. WebSocket worker 重启后 session state 怎么办？
4. 为什么模型、LM、WFST graph 必须分别版本化？
5. 如何证明新模型可以安全回滚？
6. 生产镜像为什么要使用 `uv.lock`？

<details><summary>展开参考答案</summary>

1. 精度/文本质量、性能延迟、资源与模型大小。2. 它们分别描述计算效率、交互体验和尾部稳定性。3. 明确终止并让客户端重连，或把可恢复状态放入有版本的外部状态层。4. 任一组件变化都可能改变结果，必须可追踪。5. 保留旧镜像/配置、兼容协议、灰度指标和自动回滚阈值。6. 固定可复现的依赖解析。

</details>

## 完整课程终点

至此课程覆盖：声音与特征 → 编码器 → CTC → 流式 → PGS/RTF → LM/WFST → ONNX → INT8 → HTTP/WebSocket → 容器与生产验收。下一阶段应选择真实可泛化模型和数据集，完成一次端到端工程项目。

<!-- course-upgrade-v2 -->
## 强化练习：第 30 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `容器复现`、`并发线程`、`监控灰度回滚`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**ORT 线程池与服务 worker 同时开满**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**用 wall-clock 修正并发吞吐 benchmark**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**设计量化模型的灰度和回滚门槛**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：容器复现、并发线程、监控灰度回滚。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 容器复现、并发线程、监控灰度回滚。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
